In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.svm import SVC

In [3]:
fellow = pd.read_csv("data/fellowship_data.csv")
fellow

,Name,Gender,Age,City,Graduate,Accepted
0,Kemi Bello,Female,25,Ilorin,Yes,Yes
1,Ngozi Afolabi,Female,31,Ilorin,Yes,Yes
2,Peace Afolabi,Female,30,Kano,Yes,No
3,Ngozi Afolabi,Female,35,Ilorin,Yes,Yes
4,Mariam Bello,Female,20,Ilorin,Yes,Yes
...,...,...,...,...,...,...
95,Funmi Jesu,Female,31,Ilorin,Yes,Yes
96,Bimpe Lawal,Female,21,Ilorin,Yes,Yes
97,Bayo Yusuf,Male,33,Enugu,Yes,No
98,Kunle Okafor,Male,24,Ibadan,Yes,No


In [3]:
fellow.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Name      100 non-null    object
 1   Gender    100 non-null    object
 2   Age       100 non-null    int64 
 3   City      100 non-null    object
 4   Graduate  100 non-null    object
 5   Accepted  100 non-null    object
dtypes: int64(1), object(5)
memory usage: 4.8+ KB


In [5]:
fellow.describe()

,Age
count,100.000000
mean,28.030000
std,7.038789
min,16.000000
25%,22.000000
50%,27.500000
75%,33.000000
max,45.000000


In [6]:
fellow.isnull().sum()

Name        0
Gender      0
Age         0
City        0
Graduate    0
Accepted    0
dtype: int64

In [7]:
features = ["Gender", "Age", "City", "Graduate"]
target = "Accepted"

X = fellow[features].copy()
y = fellow["Accepted"].copy()

In [8]:
X["Gender"] = X["Gender"].map({"Female": 1, "Male": 0})
X["Graduate"] = X["Graduate"].map({"No": 0, "Yes": 1})
X["City"]= X["City"].map({ 
    'Ilorin': 0,
    'Kano': 1,
    'Ibadan': 2,
    'Port Harcourt': 3,
    'Enugu': 4,
    'Abuja': 5,
    'Kaduna': 6,
    'Lagos': 7,
    'Sokoto': 8,
    'Benin City': 9,
    'Osogbo': 10
})

In [9]:
X.head()

,Gender,Age,City,Graduate
0,1,25,0,1
1,1,31,0,1
2,1,30,1,1
3,1,35,0,1
4,1,20,0,1


In [10]:
y = y.map({ 'Yes': 1, 'No': 0})

In [11]:
y.head()

0    1
1    1
2    0
3    1
4    1
Name: Accepted, dtype: int64

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42, stratify = y )

In [13]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

X_train: (80, 4)
X_test: (20, 4)


In [14]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [15]:
logistic_model = LogisticRegression(max_iter = 1000, random_state = 42)
logistic_model.fit(X_train_scaled, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [15]:
logistic_prediction = logistic_model.predict(X_test_scaled)

logistic_accuracy = accuracy_score(
    y_test,
    logistic_prediction
)
print("Logistic Regression:", logistic_accuracy)

Logistic Regression: 1.0


In [16]:
print(classification_report(y_test, logistic_prediction))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00         8
           1       1.00      1.00      1.00        12

    accuracy                           1.00        20
   macro avg       1.00      1.00      1.00        20
weighted avg       1.00      1.00      1.00        20



In [17]:
tree_model = DecisionTreeClassifier(max_depth = 3, random_state = 42)
tree_model.fit(X_train_scaled, y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,3
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [18]:
tree_prediction = tree_model.predict(X_test_scaled)

tree_accuracy = accuracy_score(
    y_test,
    tree_prediction
)
print("Decision Tree Classifier:", tree_accuracy)

Decision Tree Classifier: 1.0


In [19]:
print(classification_report(y_test, tree_prediction))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00         8
           1       1.00      1.00      1.00        12

    accuracy                           1.00        20
   macro avg       1.00      1.00      1.00        20
weighted avg       1.00      1.00      1.00        20



# SUPPORT VECTOR CLASSIFIER

In [18]:
svm_params ={
    "C": [0.1, 1, 10, 100],
    "kernel": ["linear", "rbf"],
    "gamma": ["scale", "auto"]
}
svm_grid = GridSearchCV(
    SVC(random_state = 42),
    svm_params,
    cv = 5,
    scoring = "accuracy"
)
svm_grid.fit(X_train, y_train)

,estimator,SVC(random_state=42)
,param_grid,"{'C': [0.1, 1, ...], 'gamma': ['scale', 'auto'], 'kernel': ['linear', 'rbf']}"
,scoring,'accuracy'
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,C,10


In [19]:
print("Best Parameters:")
print(svm_grid.best_params_)
print("\nBest Cross-Validation Accuracy:")
print(svm_grid.best_score_)

Best Parameters:
{'C': 10, 'gamma': 'scale', 'kernel': 'linear'}

Best Cross-Validation Accuracy:
0.9875


# TUNING SVC

In [20]:
best_svm = svm_grid.best_estimator_
tuned_svm_predictions = best_svm.predict(X_test)
tuned_svm_accuracy = accuracy_score(
 y_test,
 tuned_svm_predictions
)
print(
 "Tuned SVM Accuracy:",
 tuned_svm_accuracy * 100
)

Tuned SVM Accuracy: 100.0


## MODEL PREDICTION

In [25]:
def predict_survival(name,Gender,Age,City,Graduate):
     """
     Predict women in Ai fellowship acceptance using our model
     """
     input_data = pd.DataFrame({
     'Gender': [Gender],
     'Age': [Age],
     'City': [City],
     'Graduate': [Graduate]
     })

     prediction = best_svm.predict(input_data)[0]
     if prediction == 0:
             result = f'Wooo 😮 {name.title()}, you are Rejected 😢 '
     else:
             result = f'Cheers 🥂 {name.title()}, you Accepted 😁 '

     return result

In [26]:
print(predict_survival('Abiola', Gender=1, Age=25, City=0, Graduate=1))

Cheers 🥂 Abiola, you Accepted 😁 
